### Import Libraries
This cell loads all required libraries, including:
- `pandas` and `numpy` for data manipulation
- `sklearn` for metrics and label encoding
- `tensorflow.keras` for building and training LSTM models
- `keras_tuner` for hyperparameter tuning

In [416]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

import keras_tuner as kt

### Load data
- Read the VIF-normalized dataset from Excel
- Convert `date` column to datetime format
- Initialize empty list to track results across models

In [417]:
data_path = "../data/datasets/panel_dataset_VIF_normalized.xlsx"
df = pd.read_excel(data_path)
df['date'] = pd.to_datetime(df['date'])

all_results = []

### Initial setup
- Set the target
- Define the features by excluding targets, `date`, and `Country`

In [418]:
TARGETS = ['RECESS', 'RECESS_OVER', 'RECESS_PERIOD']
target_name = "RECESS"
exclude_cols = ['date', 'Country'] + TARGETS
feature_cols = [col for col in df.columns if col not in exclude_cols]

### Clean and encode data
- Drop rows with missing values in any of the features or the target
- Encode the `Country` column as `country_id` for modeling
- Sort by country and date to prepare for time series modeling

In [419]:
df.dropna(subset=feature_cols + [target_name], inplace=True)
df['country_id'] = LabelEncoder().fit_transform(df['Country'])
feature_cols.append('country_id')
df.sort_values(['Country', 'date'], inplace=True)

### Early Stopping Configuration
- Stop training if validation loss doesn’t improve for 10 epochs
- Restore the best model weights

In [420]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

### Data cutoffs for data split
- Define the train, validation, and test date ranges

In [421]:
train_start = "1995-01-01"
train_end = "2009-12-31"
val_start = "2010-01-01"
val_end = "2018-12-31"
test_start = "2019-01-01"
test_end = "2024-05-31"

### Split the data
- Slice the dataset into `df_train`, `df_val`, and `df_test` using the defined date ranges

In [422]:
df_train = df[(df['date'] >= train_start) & (df['date'] <= train_end)].copy()
df_val = df[(df['date'] >= val_start) & (df['date'] <= val_end)].copy()
df_test = df[(df['date'] >= test_start) & (df['date'] <= test_end)].copy()

### Sliding Window
- Set `time_steps = 24`: number of months used as input for LSTM sequences

In [423]:
time_steps = 24

### Train
- Build rolling windows per country:
  - Input: 24-month feature sequences
  - Label: target value of the 25th month

In [424]:
X_train, y_train = [], []
for _, group in df_train.groupby('Country'):
    group = group.sort_values('date').reset_index(drop=True)
    for i in range(len(group) - time_steps):
        X_train.append(group.loc[i:i+time_steps-1, feature_cols].values)
        y_train.append(group.loc[i+time_steps, target_name])

X_train = np.array(X_train)
y_train = np.array(y_train)


### Compute class weights
- Calculate weights to balance the classes for binary classification
- Helps address class imbalance in training

In [425]:
# Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))


### Validation
- Same as training, but for the validation date range

In [426]:
X_val, y_val = [], []
for _, group in df_val.groupby('Country'):
    group = group.sort_values('date').reset_index(drop=True)
    for i in range(len(group) - time_steps):
        X_val.append(group.loc[i:i+time_steps-1, feature_cols].values)
        y_val.append(group.loc[i+time_steps, target_name])

X_val = np.array(X_val)
y_val = np.array(y_val)

### Testing
- Same process again for the test date range

In [427]:
X_test, y_test = [], []
for _, group in df_test.groupby('Country'):
    group = group.sort_values('date').reset_index(drop=True)
    for i in range(len(group) - time_steps):
        X_test.append(group.loc[i:i+time_steps-1, feature_cols].values)
        y_test.append(group.loc[i+time_steps, target_name])

X_test = np.array(X_test)
y_test = np.array(y_test)

### Display split sizes
- Print number of training, validation, and test sequences

In [428]:
print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

Train: 686 samples
Validation: 504 samples
Test: 240 samples


### Define Standard LSTM Builder for Keras Tuner
This function constructs a single-layer LSTM model for binary classification.
- `units`, `dropout`, `recurrent_dropout`, and `optimizer` are treated as tunable hyperparameters.
- The final output uses a sigmoid activation for binary classification.

In [429]:
def build_standard_lstm(hp):
    model = Sequential()
    model.add(LSTM(
        units=hp.Choice('units', [64, 128, 256]),
        input_shape=(X_train.shape[1], X_train.shape[2]),
        return_sequences=False,
        recurrent_dropout=hp.Float('rec_dropout', 0.0, 0.3, step=0.1)
    ))
    model.add(Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'rmsprop']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


### Define Bidirectional LSTM Builder
Similar to the standard LSTM, but wraps the LSTM layer in a `Bidirectional` wrapper to allow learning from both past and future time steps. This often improves performance on sequence prediction tasks.

In [430]:
def build_bidirectional_lstm(hp):
    model = Sequential()
    model.add(Bidirectional(LSTM(
        units=hp.Choice('units', [64, 128, 256]),
        recurrent_dropout=hp.Float('rec_dropout', 0.0, 0.3, step=0.1)
    ), input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'rmsprop']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

### Define Stacked LSTM Builder
Creates a two-layer LSTM architecture:
- The first LSTM returns sequences to feed into the second layer.
- Adds dropout after each LSTM layer.
- The output is passed through a sigmoid layer for binary classification.


In [431]:
def build_stacked_lstm(hp):
    model = Sequential()
    model.add(LSTM(
        units=hp.Choice('units_1', [64, 128, 256]),
        return_sequences=True,
        input_shape=(X_train.shape[1], X_train.shape[2]),
        recurrent_dropout=hp.Float('rec_dropout_1', 0.0, 0.3, step=0.1)
    ))
    model.add(Dropout(hp.Float('dropout_1', 0.2, 0.5, step=0.1)))

    model.add(LSTM(
        units=hp.Choice('units_2', [64, 128]),
        return_sequences=False,
        recurrent_dropout=hp.Float('rec_dropout_2', 0.0, 0.3, step=0.1)
    ))
    model.add(Dropout(hp.Float('dropout_2', 0.2, 0.5, step=0.1)))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'rmsprop']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

### Model Builder Dictionary
- Maps string names to the corresponding model-building functions.
-cAllows easy iteration over different architectures in the next step.


In [432]:
model_builders = {
    "standard": build_standard_lstm,
    "bidirectional": build_bidirectional_lstm,
    "stacked": build_stacked_lstm
}

### Define Targets and Model Types
Specifies the three binary targets and the three LSTM architectures to iterate over for hyperparameter tuning.


In [433]:
targets = ["RECESS", "RECESS_OVER", "RECESS_PERIOD"]
model_types = ["standard", "bidirectional", "stacked"]

### F1 Score Evaluator
Evaluates a trained model on the validation set by:
- Scanning thresholds between 0 and 1.
- Returning the maximum F1 score found.
Used for final model selection instead of accuracy.

In [434]:
def score_f1(model):
    val_pred = model.predict(X_val).flatten()
    _, _, thresholds = precision_recall_curve(y_val, val_pred)
    f1s = [f1_score(y_val, (val_pred >= t).astype(int)) for t in thresholds]
    return max(f1s)

### Main Tuning Loop for All Models and Targets
Loops over:
1. Each target (`RECESS`, `RECESS_OVER`, `RECESS_PERIOD`)
2. Each LSTM architecture (`standard`, `bidirectional`, `stacked`)

For each combination:
- Rebuilds y-train, y-val, and y-test using the new target.
- Runs `keras_tuner.RandomSearch` for 10 trials.
- Selects the best model using **F1 score**, even though tuning was done on **validation accuracy**.
- Prints best hyperparameters for each tuned model.

In [435]:
for target_name in targets:
    print(f"\nTuning models for target: {target_name}\n")

    df.dropna(subset=feature_cols + [target_name], inplace=True)

    y_train = []
    for _, group in df_train.groupby('Country'):
        group = group.sort_values('date').reset_index(drop=True)
        for i in range(len(group) - time_steps):
            y_train.append(group.loc[i+time_steps, target_name])
    y_train = np.array(y_train)

    y_val = []
    for _, group in df_val.groupby('Country'):
        group = group.sort_values('date').reset_index(drop=True)
        for i in range(len(group) - time_steps):
            y_val.append(group.loc[i+time_steps, target_name])
    y_val = np.array(y_val)

    y_test = []
    for _, group in df_test.groupby('Country'):
        group = group.sort_values('date').reset_index(drop=True)
        for i in range(len(group) - time_steps):
            y_test.append(group.loc[i+time_steps, target_name])
    y_test = np.array(y_test)

    for model_type in model_types:
        print(f"Tuning {model_type} model on {target_name}")

        tuner = kt.RandomSearch(
            model_builders[model_type],
            objective=kt.Objective("val_accuracy", direction="max"),  # Still needed for training logs
            max_trials=10,
            executions_per_trial=1,
            directory='tuner_logs',
            project_name=f"tune_{model_type}_lstm_{target_name}"
        )

        tuner.search(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=32,
            callbacks=[early_stop],
            class_weight=class_weights,
            verbose=0
        )

        # Now score models using F1
        best_model = max(tuner.get_best_models(num_models=5), key=score_f1)
        best_hps = tuner.get_best_hyperparameters(1)[0]

        print(f"Done tuning {model_type} — {target_name}")
        print("Best Hyperparameters:", best_hps.values)


🔄 Tuning models for target: RECESS

✨ Tuning standard model on RECESS
Reloading Tuner from tuner_logs\tune_standard_lstm_RECESS\tuner0.json


C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 7 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 12 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step
✅ Done tuning standard — RECESS
Best Hyperparameters: {'units': 64, 'rec_dropout': 0.1, 'dropout': 0.4, 'optimizer': 'rmsprop'}
✨ Tuning bidirectional model on RECESS
Reloading Tuner from tuner_logs\tune_bidirectional_lstm_RECESS\tuner0.json


C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
✅ Done tuning bidirectional — RECESS
Best Hyperparameters: {'units': 64, 'rec_dropout': 0.2, 'dropout': 0.30000000000000004, 'optimizer': 'rmsprop'}
✨ Tuning stacked model on RECESS
Reloading Tuner from tuner_logs\tune_stacked_lstm_RECESS\tuner0.json


...

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step
✅ Done tuning standard — RECESS_PERIOD
Best Hyperparameters: {'units': 64, 'rec_dropout': 0.2, 'dropout': 0.4, 'optimizer': 'adam'}
✨ Tuning bidirectional model on RECESS_PERIOD
Reloading Tuner from tuner_logs\tune_bidirectional_lstm_RECESS_PERIOD\tuner0.json


C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step
✅ Done tuning bidirectional — RECESS_PERIOD
Best Hyperparameters: {'units': 128, 'rec_dropout': 0.2, 'dropout': 0.30000000000000004, 'optimizer': 'rmsprop'}
✨ Tuning stacked model on RECESS_PERIOD
Reloading Tuner from tuner_logs\tune_stacked_lstm_RECESS_PERIOD\tuner0.json


C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step
✅ Done tuning stacked — RECESS_PERIOD
Best Hyperparameters: {'units_1': 64, 'rec_dropout_1': 0.1, 'dropout_1': 0.30000000000000004, 'units_2': 128, 'rec_dropout_2': 0.1, 'dropout_2': 0.30000000000000004, 'optimizer': 'adam'}


### Define Focal Loss Function
This custom loss is designed to address extreme class imbalance.
- `alpha`: scales the loss to focus more on the minority class.
- `gamma`: controls how much the function focuses on hard-to-classify examples.
Although defined, this focal loss is **not actually used** in the models below.


In [437]:
def focal_loss(gamma=2., alpha=0.25):
    def loss_fn(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        return -alpha * tf.pow(1. - pt, gamma) * tf.math.log(pt)
    return loss_fn

### Build Standard LSTM Model for RECESS Target
Single-layer LSTM with 64 units, dropout for regularization, and a sigmoid output for binary prediction.

In [438]:
model_standard_recess = Sequential()
model_standard_recess.add(LSTM(
    units=64,
    recurrent_dropout=0.1,
    input_shape=(X_train.shape[1], X_train.shape[2]),
    return_sequences=False
))
model_standard_recess.add(Dropout(0.4))
model_standard_recess.add(Dense(1, activation='sigmoid'))

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Compile the Model
Using:
- `rmsprop`: well-suited for time-series RNN training.
- `binary_crossentropy`: appropriate for binary classification.

In [439]:
model_standard_recess.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Train the Model
- Uses early stopping to prevent overfitting.
- Applies class weights to balance training on imbalanced targets.

In [440]:
history_standard = model_standard_recess.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    class_weight=class_weights,
    verbose=1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.7911 - loss: 0.4790 - val_accuracy: 0.6786 - val_loss: 0.5391
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9012 - loss: 0.3411 - val_accuracy: 0.5952 - val_loss: 0.6112
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8904 - loss: 0.3083 - val_accuracy: 0.6012 - val_loss: 0.6075
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9118 - loss: 0.2410 - val_accuracy: 0.4821 - val_loss: 0.7809
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9088 - loss: 0.2064 - val_accuracy: 0.5476 - val_loss: 0.7543
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9361 - loss: 0.1861 - val_accuracy: 0.6310 - val_loss: 0.6616
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9520 - loss: 0.1533 - val_accuracy: 0.4980 - val_loss: 0.8393
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9549 - loss: 0.1316 - val_accuracy: 0.

### Predict on Validation Set (for threshold tuning)

In [441]:
y_val_prob = model_standard_recess.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step


### Threshold Tuning
Scans all decision thresholds to find the one that maximizes the F1 score on the validation set.

In [442]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)

# Skip the last precision/recall which has no corresponding threshold
f1s = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)

best_idx = np.argmax(f1s)
best_thresh = thresholds[best_idx]
best_f1 = f1s[best_idx]

print(f"[TUNING] Best threshold: {best_thresh:.2f} — F1: {best_f1:.4f}")

[TUNING] Best threshold: 0.25 — F1: 0.1497


### Apply Tuned Threshold on Test Predictions
Converts probabilities to binary outputs using the best threshold.

In [443]:
y_pred = model_standard_recess.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


### Evaluate Model Performance on Test Set
Computes standard classification metrics including AUC, precision, recall, F1, and accuracy.

In [444]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

In [445]:
print(f"[RESULTS] RECESS — Standard LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS — Standard LSTM
AUC:       0.6847
F1 Score:  0.1674
Precision: 0.0913
Recall:    1.0000
Accuracy:  0.2125
Threshold: 0.24601233005523682


### Save Results to Shared List
Adds performance metrics for this model configuration to `all_results`.

In [446]:
all_results.append({
    "Model": "Standard LSTM",
    "Target": "RECESS",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

### Build Bidirectional LSTM for RECESS
- This is the actual BiLSTM model used for evaluation.
- Includes one bidirectional layer and moderate dropout.

In [448]:
model_bi_recess = Sequential()
model_bi_recess.add(Bidirectional(LSTM(
    units=64,
    recurrent_dropout=0.2
), input_shape=(X_train.shape[1], X_train.shape[2])))
model_bi_recess.add(Dropout(0.30000000000000004))
model_bi_recess.add(Dense(1, activation='sigmoid'))

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Compile BiLSTM Model
- Same compile settings as the standard LSTM.


In [1]:
model_bi_recess.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

NameError: name 'model_bi_recess' is not defined

### Train BiLSTM Model
- Same training setup: early stopping, 100 epochs max, 32 batch size.


In [450]:
history_bi = model_bi_recess.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 72ms/step - accuracy: 0.7493 - loss: 0.5123 - val_accuracy: 0.9206 - val_loss: 0.3444
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.8646 - loss: 0.3432 - val_accuracy: 0.7817 - val_loss: 0.3976
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8970 - loss: 0.2734 - val_accuracy: 0.7619 - val_loss: 0.4094
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9025 - loss: 0.2314 - val_accuracy: 0.6964 - val_loss: 0.4700
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.8987 - loss: 0.2162 - val_accuracy: 0.6607 - val_loss: 0.5241
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9114 - loss: 0.2001 - val_accuracy: 0.6627 - val_loss: 0.5763
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9165 - loss: 0.1781 - val_accuracy: 0.6488 - val_loss: 0.6298
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9348 - loss: 0.1452 - val_accuracy: 0.

### Threshold Tuning for BiLSTM
Finds best F1-maximizing threshold on validation set, just like for standard LSTM.

In [451]:
y_val_prob = model_bi_recess.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step


In [452]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)

f1s = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)

best_idx = np.argmax(f1s)
best_thresh = thresholds[best_idx]
best_f1 = f1s[best_idx]

print(f"[TUNING] Best threshold: {best_thresh:.2f} — F1: {best_f1:.4f}")

[TUNING] Best threshold: 0.12 — F1: 0.1392


### Make Predictions on Test Set and Apply Threshold

In [453]:
y_pred = model_bi_recess.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


### Evaluate BiLSTM Test Performance

In [454]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

### Display Results for BiLSTM

In [455]:
print(f"[RESULTS] RECESS — Bidirectional LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS — Bidirectional LSTM
AUC:       0.6990
F1 Score:  0.1652
Precision: 0.0900
Recall:    1.0000
Accuracy:  0.2000
Threshold: 0.1188676729798317


In [456]:
all_results.append({
    "Model": "Bidirectional LSTM",
    "Target": "RECESS",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

### Build Stacked LSTM for RECESS
This model contains two LSTM layers stacked on top of each other:
- The first LSTM layer (128 units) returns full sequences to feed into the second.
- The second LSTM also has 128 units and outputs a single vector.
- Dropout is used between layers to reduce overfitting.
- Final layer is a sigmoid neuron for binary classification.

In [457]:
model_stacked_recess = Sequential()
model_stacked_recess.add(LSTM(
    units=128,
    return_sequences=True,
    recurrent_dropout=0.1,  # Try 0.0
    input_shape=(X_train.shape[1], X_train.shape[2])
))
model_stacked_recess.add(Dropout(0.2))  # Try 0.4
model_stacked_recess.add(LSTM(
    units=128,   # Try 128
    return_sequences=False,
    recurrent_dropout=0.2   # Try 0.1
))
model_stacked_recess.add(Dropout(0.30000000000000004)) # Try 0.30000000000000004
model_stacked_recess.add(Dense(1, activation='sigmoid'))

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Compile the Stacked LSTM
Using:
- `rmsprop`: common for sequence models.
- `binary_crossentropy`: appropriate for binary classification tasks like RECESS detection.

In [458]:
model_stacked_recess.compile(
    optimizer='rmsprop', # Try rmsprop
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Train the Model
- Up to 100 epochs, but early stopping is used to restore the best model.
- Mini-batch size is 32.

In [459]:
history_stacked = model_stacked_recess.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 81ms/step - accuracy: 0.7799 - loss: 0.4892 - val_accuracy: 0.6190 - val_loss: 0.5502
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.8903 - loss: 0.3065 - val_accuracy: 0.7004 - val_loss: 0.5254
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.8909 - loss: 0.2637 - val_accuracy: 0.6329 - val_loss: 0.7062
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.9139 - loss: 0.2026 - val_accuracy: 0.5258 - val_loss: 1.0473
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9137 - loss: 0.1966 - val_accuracy: 0.4921 - val_loss: 1.1233
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.9372 - loss: 0.1539 - val_accuracy: 0.6111 - val_loss: 0.9715
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9290 - loss: 0.1642 - val_accuracy: 0.6369 - val_loss: 0.8515
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9479 - loss: 0.1389 - val_accuracy: 0.

### Predict on Validation Set
Start with default threshold of 0.5, but this will be tuned.

In [460]:
y_val_prob = model_stacked_recess.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step


### Optimize Threshold for F1 Score
Scans a range of threshold values to find the one that yields the highest F1 score.
- This is crucial in imbalanced datasets like recession prediction.

In [461]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)

# Skip the last precision/recall which has no corresponding threshold
f1s = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)

best_idx = np.argmax(f1s)
best_thresh = thresholds[best_idx]
best_f1 = f1s[best_idx]

print(f"[TUNING] Best threshold: {best_thresh:.2f} — F1: {best_f1:.4f}")

[TUNING] Best threshold: 0.12 — F1: 0.1765


### Apply Tuned Threshold on Test Set
Predicted probabilities are converted to class labels using the tuned threshold.

In [462]:
y_pred = model_stacked_recess.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


### Evaluate Final Performance on Test Set
Key metrics:
- AUC: how well the model ranks positive vs. negative cases.
- F1: harmonic mean of precision and recall.
- Accuracy: overall correct predictions.

In [463]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

In [464]:
print(f"[RESULTS] RECESS — Stacked LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS — Stacked LSTM
AUC:       0.7221
F1 Score:  0.1751
Precision: 0.0960
Recall:    1.0000
Accuracy:  0.2542
Threshold: 0.12208790332078934


### Log Results
Appends the current model's evaluation results to the `all_results` list for later summarization.

In [465]:
all_results.append({
    "Model": "Stacked LSTM",
    "Target": "RECESS",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

In [466]:
results_df = pd.DataFrame(all_results)
results_df

,Model,Target,AUC,F1,Precision,Recall,Accuracy,Threshold
0,Standard LSTM,RECESS,0.684687,0.167401,0.091346,1.0,0.212500,0.246012
1,Bidirectional LSTM,RECESS,0.698976,0.165217,0.090047,1.0,0.200000,0.118868
2,Stacked LSTM,RECESS,0.722077,0.175115,0.095960,1.0,0.254167,0.122088


### Update Target
Set the target variable to `RECESS_OVER`, which identifies the conclusion of a recession period.

### RECESS_OVER

In [467]:
target_name = "RECESS_OVER"

### Preview Class Distribution
Show the number of positive and negative examples for the current target in the training set.


In [468]:
print("Training target preview:", np.unique(y_train, return_counts=True))
print("Currently training on:", target_name)

Training target preview: (array([0, 1]), array([559, 127]))
Currently training on: RECESS_OVER


### Standard LSTM Model for RECESS_OVER
Build a single-layer LSTM model:
- 64 units, no recurrent dropout
- Dropout to mitigate overfitting
- Sigmoid output for binary classification


In [469]:
model_standard_over = Sequential()
model_standard_over.add(LSTM(64, recurrent_dropout=0.0, return_sequences=False,
                             input_shape=(X_train.shape[1], X_train.shape[2])))
model_standard_over.add(Dropout(0.2))
model_standard_over.add(Dense(1, activation='sigmoid'))

model_standard_over.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Train Standard LSTM
Train the model for up to 100 epochs using early stopping to prevent overfitting.


In [470]:
history_standard_over = model_standard_over.fit(
X_train, y_train, validation_data=(X_val, y_val),
epochs=100, batch_size=32, callbacks=[early_stop], verbose=1)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.7790 - loss: 0.5028 - val_accuracy: 0.7817 - val_loss: 0.4146
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8775 - loss: 0.3330 - val_accuracy: 0.6746 - val_loss: 0.4998
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9045 - loss: 0.2568 - val_accuracy: 0.6567 - val_loss: 0.5178
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9167 - loss: 0.2354 - val_accuracy: 0.6528 - val_loss: 0.5440
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9390 - loss: 0.1849 - val_accuracy: 0.5853 - val_loss: 0.6369
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9384 - loss: 0.1753 - val_accuracy: 0.6409 - val_loss: 0.6043
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9514 - loss: 0.1514 - val_accuracy: 0.6667 - val_loss: 0.6039
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9463 - loss: 0.1299 - val_accuracy: 0.

### Predict and Initialize Threshold
Get predicted probabilities for validation data and initialize threshold optimization variables.

In [471]:
y_val_prob = model_standard_over.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


### Threshold Tuning (Validation)
Find the best threshold that maximizes F1 score on the validation set.


In [472]:
for t in np.arange(0.1, 0.9, 0.01):
    y_val_pred = (y_val_prob >= t).astype(int)
    f1_val = f1_score(y_val, y_val_pred)
    if f1_val > best_f1:
        best_f1 = f1_val
        best_thresh = t
print(f"[TUNING] Best threshold for Standard LSTM (RECESS_OVER): {best_thresh:.2f}")

[TUNING] Best threshold for Standard LSTM (RECESS_OVER): 0.10


### Test Set Prediction
Apply the tuned threshold to convert test predictions into binary class labels.


In [473]:
y_pred = model_standard_over.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


### Evaluation Metrics (Standard LSTM)
Calculate all standard classification metrics on the test set.


In [474]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

### Display Results
Show model performance using all metrics including the optimized threshold.


In [475]:
print(f"[RESULTS] RECESS_OVER — Standard LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS_OVER — Standard LSTM
AUC:       0.6628
F1 Score:  0.1645
Precision: 0.0896
Recall:    1.0000
Accuracy:  0.1958
Threshold: 0.1


### Log Results
Store model performance in the `all_results` list for future comparison.


In [476]:
all_results.append({
    "Model": "Standard LSTM",
    "Target": "RECESS_OVER",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

### Bidirectional LSTM Model for RECESS_OVER
Creates a Bidirectional LSTM with:
- 64 units and dropout
- RMSprop optimizer and binary loss
- Suitable for capturing forward and backward sequence dynamics

In [477]:
model_bi_over = Sequential()
model_bi_over.add(Bidirectional(LSTM(64, recurrent_dropout=0.2),
                                input_shape=(X_train.shape[1], X_train.shape[2])))
model_bi_over.add(Dropout(0.30000000000000004))
model_bi_over.add(Dense(1, activation='sigmoid'))
model_bi_over.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Train Bidirectional LSTM
Same setup as previous model: early stopping, 100 epochs max.


In [478]:
history_bi_over = model_bi_over.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=100, batch_size=32, callbacks=[early_stop], verbose=1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 73ms/step - accuracy: 0.7765 - loss: 0.5102 - val_accuracy: 0.7341 - val_loss: 0.4434
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8615 - loss: 0.3204 - val_accuracy: 0.6885 - val_loss: 0.4829
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8791 - loss: 0.2752 - val_accuracy: 0.5794 - val_loss: 0.6505
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9139 - loss: 0.2343 - val_accuracy: 0.6270 - val_loss: 0.5934
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.9185 - loss: 0.1996 - val_accuracy: 0.6071 - val_loss: 0.6889
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9503 - loss: 0.1484 - val_accuracy: 0.6032 - val_loss: 0.7858
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9342 - loss: 0.1605 - val_accuracy: 0.6290 - val_loss: 0.7521
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9213 - loss: 0.1755 - val_accuracy: 0.

### Prepare for Threshold Tuning
Flatten predictions and reset best score trackers.


In [479]:
y_val_prob = model_bi_over.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step


### Tune Threshold (Bidirectional)
Same as before, now applied to the bidirectional model.

In [480]:
for t in np.arange(0.1, 0.9, 0.01):
    y_val_pred = (y_val_prob >= t).astype(int)
    f1_val = f1_score(y_val, y_val_pred)
    if f1_val > best_f1:
        best_f1 = f1_val
        best_thresh = t
print(f"[TUNING] Best threshold for Bidirectional LSTM (RECESS_OVER): {best_thresh:.2f}")

[TUNING] Best threshold for Bidirectional LSTM (RECESS_OVER): 0.23


### Predict on Test Set
Use the optimal threshold on test predictions.


In [481]:
y_pred = model_bi_over.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


### Evaluation Metrics (Bidirectional LSTM)
Assess final performance of the bidirectional model.

In [482]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

### Display Results (Bidirectional)
Print performance for easy comparison with other models.


In [483]:
print(f"[RESULTS] RECESS_OVER — Bidirectional LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS_OVER — Bidirectional LSTM
AUC:       0.6633
F1 Score:  0.1776
Precision: 0.0974
Recall:    1.0000
Accuracy:  0.2667
Threshold: 0.22999999999999995


### Save Results (Bidirectional)
Append performance to tracking list.

In [484]:
all_results.append({
    "Model": "Bidirectional LSTM",
    "Target": "RECESS_OVER",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

### Stacked LSTM Model for RECESS_OVER
Two LSTM layers stacked:
- First returns sequences, second outputs final representation.
- Added dropout for regularization.
- Adam optimizer used here.

In [485]:
model_stacked_over = Sequential()
model_stacked_over.add(LSTM(64, return_sequences=True, recurrent_dropout=0.1,
                            input_shape=(X_train.shape[1], X_train.shape[2])))
model_stacked_over.add(Dropout(0.30000000000000004))
model_stacked_over.add(LSTM(64, return_sequences=False, recurrent_dropout=0.0))
model_stacked_over.add(Dropout(0.4))
model_stacked_over.add(Dense(1, activation='sigmoid'))
model_stacked_over.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Train Stacked LSTM
Train model with validation monitoring and early stopping.


In [486]:
history_stacked_over = model_stacked_over.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=100, batch_size=32, callbacks=[early_stop], verbose=1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.7386 - loss: 0.5376 - val_accuracy: 0.9008 - val_loss: 0.3606
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.8483 - loss: 0.3568 - val_accuracy: 0.7837 - val_loss: 0.4107
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.8907 - loss: 0.2857 - val_accuracy: 0.7143 - val_loss: 0.4805
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9283 - loss: 0.1916 - val_accuracy: 0.7044 - val_loss: 0.4972
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9320 - loss: 0.1664 - val_accuracy: 0.6607 - val_loss: 0.5339
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9423 - loss: 0.1401 - val_accuracy: 0.6468 - val_loss: 0.6066
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9300 - loss: 0.1900 - val_accuracy: 0.6627 - val_loss: 0.5809
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.9409 - loss: 0.1382 - val_accuracy: 0.

### Predict and Initialize
Make predictions and prepare to find best threshold.


In [487]:
y_val_prob = model_stacked_over.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step


### Tune Threshold (Stacked)
Find best threshold for F1 on the validation set.


In [488]:
for t in np.arange(0.1, 0.9, 0.01):
    y_val_pred = (y_val_prob >= t).astype(int)
    f1_val = f1_score(y_val, y_val_pred)
    if f1_val > best_f1:
        best_f1 = f1_val
        best_thresh = t
print(f"[TUNING] Best threshold for Stacked LSTM (RECESS_OVER): {best_thresh:.2f}")

[TUNING] Best threshold for Stacked LSTM (RECESS_OVER): 0.11


### Apply Tuned Threshold to Test Predictions


In [489]:
y_pred = model_stacked_over.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


### Final Evaluation (Stacked)
Compute performance metrics for Stacked LSTM.


In [490]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

### Display Results (Stacked)
Output the final evaluation metrics for this architecture.


In [491]:
print(f"[RESULTS] RECESS_OVER — Stacked LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS_OVER — Stacked LSTM
AUC:       0.7397
F1 Score:  0.1645
Precision: 0.0896
Recall:    1.0000
Accuracy:  0.1958
Threshold: 0.11


### Save Results
Append Stacked LSTM performance to results list.


In [492]:
all_results.append({
    "Model": "Stacked LSTM",
    "Target": "RECESS_OVER",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

### RECESS_PERIOD

### Update Target
Switch the target variable to `RECESS_PERIOD`, which indicates whether a month falls anywhere within a recession.

In [493]:
target_name = "RECESS_PERIOD"

### Check Target Class Balance
Print how many positive and negative samples are present for RECESS_PERIOD in the training data.

In [494]:
print("Training target preview:", np.unique(y_train, return_counts=True))
print("Currently training on:", target_name)

Training target preview: (array([0, 1]), array([559, 127]))
Currently training on: RECESS_PERIOD


### Standard LSTM Model for RECESS_PERIOD
Define a single-layer LSTM with:
- 64 units, dropout regularization
- Sigmoid output for binary classification

In [495]:
model_standard_period = Sequential()
model_standard_period.add(LSTM(64, recurrent_dropout=0.2, return_sequences=False,
                             input_shape=(X_train.shape[1], X_train.shape[2])))
model_standard_period.add(Dropout(0.4))
model_standard_period.add(Dense(1, activation='sigmoid'))

model_standard_period.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Train Standard LSTM
Train the model using early stopping to prevent overfitting.


In [496]:
history_standard_period = model_standard_period.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=100, batch_size=32, callbacks=[early_stop], verbose=1)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.6761 - loss: 0.5986 - val_accuracy: 0.8175 - val_loss: 0.3972
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8477 - loss: 0.3518 - val_accuracy: 0.7698 - val_loss: 0.4543
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8847 - loss: 0.3234 - val_accuracy: 0.7063 - val_loss: 0.4958
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9092 - loss: 0.2859 - val_accuracy: 0.6706 - val_loss: 0.5223
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9033 - loss: 0.2788 - val_accuracy: 0.6726 - val_loss: 0.5255
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9109 - loss: 0.2257 - val_accuracy: 0.6171 - val_loss: 0.6095
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9309 - loss: 0.1823 - val_accuracy: 0.6587 - val_loss: 0.5953
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9248 - loss: 0.1825 - val_accuracy: 0.

### Initialize Threshold Search
Obtain predicted probabilities on validation set and initialize variables to tune the classification threshold.

In [497]:
y_val_prob = model_standard_period.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step


### Tune Threshold for Best F1 Score
Search over thresholds and choose the one that maximizes F1 on validation set.


In [498]:
for t in np.arange(0.1, 0.9, 0.01):
    y_val_pred = (y_val_prob >= t).astype(int)
    f1_val = f1_score(y_val, y_val_pred)
    if f1_val > best_f1:
        best_f1 = f1_val
        best_thresh = t
print(f"[TUNING] Best threshold for Standard LSTM (RECESS_PERIOD): {best_thresh:.2f}")

[TUNING] Best threshold for Standard LSTM (RECESS_PERIOD): 0.15


### Generate Final Predictions (Standard LSTM)
Apply tuned threshold to get binary predictions on the test set.


In [499]:
y_pred = model_standard_period.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


### Evaluate Model (Standard LSTM)
Compute performance metrics on the test set.


In [500]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

### Display Results (Standard LSTM)
Print the model performance metrics for this architecture.


In [501]:
print(f"[RESULTS] RECESS_PERIOD — Standard LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS_PERIOD — Standard LSTM
AUC:       0.6678
F1 Score:  0.1645
Precision: 0.0896
Recall:    1.0000
Accuracy:  0.1958
Threshold: 0.14999999999999997


### Save Results (Standard LSTM)
Store metrics for RECESS_PERIOD in the `all_results` list.


In [502]:
all_results.append({
    "Model": "Standard LSTM",
    "Target": "RECESS_PERIOD",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

### Bidirectional LSTM Model for RECESS_PERIOD
Use a larger 128-unit bidirectional LSTM to capture both forward and backward temporal dependencies.


In [503]:
model_bi_period = Sequential()
model_bi_period.add(Bidirectional(LSTM(128, recurrent_dropout=0.2),
                                input_shape=(X_train.shape[1], X_train.shape[2])))
model_bi_period.add(Dropout(0.30000000000000004 ))
model_bi_period.add(Dense(1, activation='sigmoid'))
model_bi_period.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Train Bidirectional LSTM
Train model with validation monitoring and early stopping.


In [504]:
history_bi_period = model_bi_period.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=100, batch_size=32, callbacks=[early_stop], verbose=1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - accuracy: 0.7814 - loss: 0.4929 - val_accuracy: 0.7937 - val_loss: 0.4247
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.8962 - loss: 0.2899 - val_accuracy: 0.7004 - val_loss: 0.5100
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.8937 - loss: 0.2498 - val_accuracy: 0.6389 - val_loss: 0.6667
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9282 - loss: 0.1814 - val_accuracy: 0.5774 - val_loss: 0.8568
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9405 - loss: 0.1541 - val_accuracy: 0.5536 - val_loss: 0.9717
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9401 - loss: 0.1321 - val_accuracy: 0.4444 - val_loss: 1.3361
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9342 - loss: 0.1595 - val_accuracy: 0.5714 - val_loss: 0.9820
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9395 - loss: 0.1346 - val_accuracy: 0.

### Predict and Initialize Threshold Search
Prepare for tuning by flattening predictions and initializing tracking variables.


In [505]:
y_val_prob = model_bi_period.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step


### Threshold Tuning (Bidirectional)
Find best decision threshold to optimize F1 score.


In [506]:
for t in np.arange(0.1, 0.9, 0.01):
    y_val_pred = (y_val_prob >= t).astype(int)
    f1_val = f1_score(y_val, y_val_pred)
    if f1_val > best_f1:
        best_f1 = f1_val
        best_thresh = t
print(f"[TUNING] Best threshold for Bidirectional LSTM (RECESS_PERIOD): {best_thresh:.2f}")

[TUNING] Best threshold for Bidirectional LSTM (RECESS_PERIOD): 0.15


### Apply Threshold to Test Predictions (Bidirectional)


In [507]:
y_pred = model_bi_period.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


### Evaluate Bidirectional Model
Compute test set performance.


In [508]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

### Print Results (Bidirectional LSTM)


In [509]:
print(f"[RESULTS] RECESS_PERIOD — Bidirectional LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS_PERIOD — Bidirectional LSTM
AUC:       0.7514
F1 Score:  0.1712
Precision: 0.0936
Recall:    1.0000
Accuracy:  0.2333
Threshold: 0.14999999999999997


### Save Results (Bidirectional LSTM)


In [510]:
all_results.append({
    "Model": "Bidirectional LSTM",
    "Target": "RECESS_PERIOD",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

### Stacked LSTM Model for RECESS_PERIOD
Define a two-layer LSTM where the first layer returns full sequences and the second summarizes them.

In [511]:
model_stacked_period = Sequential()
model_stacked_period.add(LSTM(64, return_sequences=True, recurrent_dropout=0.1,
                            input_shape=(X_train.shape[1], X_train.shape[2])))
model_stacked_period.add(Dropout(0.30000000000000004))
model_stacked_period.add(LSTM(128, return_sequences=False, recurrent_dropout=0.1))
model_stacked_period.add(Dropout(0.30000000000000004))
model_stacked_period.add(Dense(1, activation='sigmoid'))
model_stacked_period.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Train Stacked LSTM

In [512]:
history_stacked_period = model_stacked_period.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=100, batch_size=32, callbacks=[early_stop], verbose=1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 0.8112 - loss: 0.4951 - val_accuracy: 0.8115 - val_loss: 0.4098
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9037 - loss: 0.2799 - val_accuracy: 0.6845 - val_loss: 0.4896
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.8925 - loss: 0.2657 - val_accuracy: 0.6210 - val_loss: 0.6790
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9550 - loss: 0.1545 - val_accuracy: 0.6409 - val_loss: 0.7209
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9161 - loss: 0.1949 - val_accuracy: 0.6091 - val_loss: 0.7304
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.9310 - loss: 0.1470 - val_accuracy: 0.5893 - val_loss: 0.9362
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9258 - loss: 0.1750 - val_accuracy: 0.5972 - val_loss: 0.7987
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9332 - loss: 0.1213 - val_accuracy: 0.

### Predict Validation and Initialize Threshold

In [513]:
y_val_prob = model_stacked_period.predict(X_val).flatten()
best_thresh = 0.5
best_f1 = 0

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step


### Find Best Threshold (Stacked)


In [514]:
for t in np.arange(0.1, 0.9, 0.01):
    y_val_pred = (y_val_prob >= t).astype(int)
    f1_val = f1_score(y_val, y_val_pred)
    if f1_val > best_f1:
        best_f1 = f1_val
        best_thresh = t
print(f"[TUNING] Best threshold for Stacked LSTM (RECESS_PERIOD): {best_thresh:.2f}")

[TUNING] Best threshold for Stacked LSTM (RECESS_PERIOD): 0.70


### Generate Binary Test Predictions (Stacked)


In [515]:
y_pred = model_stacked_period.predict(X_test).flatten()
y_pred_binary = (y_pred >= best_thresh).astype(int)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


### Final Metrics (Stacked LSTM)


In [516]:
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
accuracy = accuracy_score(y_test, y_pred_binary)

### Print Model Performance (Stacked)


In [517]:
print(f"[RESULTS] RECESS_PERIOD — Stacked LSTM")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Threshold: {best_thresh}")

[RESULTS] RECESS_PERIOD — Stacked LSTM
AUC:       0.7218
F1 Score:  0.1967
Precision: 0.1165
Recall:    0.6316
Accuracy:  0.5917
Threshold: 0.6999999999999996


### Save Final Results


In [518]:
all_results.append({
    "Model": "Stacked LSTM",
    "Target": "RECESS_PERIOD",
    "AUC": auc,
    "F1": f1,
    "Precision": precision,
    "Recall": recall,
    "Accuracy": accuracy,
    "Threshold": best_thresh
})

### Compile Results into DataFrame
Aggregate all performance metrics across models and targets for comparison.

In [519]:
results_df = pd.DataFrame(all_results)
results_df

,Model,Target,AUC,F1,Precision,Recall,Accuracy,Threshold
0,Standard LSTM,RECESS,0.684687,0.167401,0.091346,1.000000,0.212500,0.246012
1,Bidirectional LSTM,RECESS,0.698976,0.165217,0.090047,1.000000,0.200000,0.118868
2,Stacked LSTM,RECESS,0.722077,0.175115,0.095960,1.000000,0.254167,0.122088
3,Standard LSTM,RECESS_OVER,0.662777,0.164502,0.089623,1.000000,0.195833,0.100000
4,Bidirectional LSTM,RECESS_OVER,0.663253,0.177570,0.097436,1.000000,0.266667,0.230000
5,Stacked LSTM,RECESS_OVER,0.739700,0.164502,0.089623,1.000000,0.195833,0.110000
6,Standard LSTM,RECESS_PERIOD,0.667778,0.164502,0.089623,1.000000,0.195833,0.150000
7,Bidirectional LSTM,RECESS_PERIOD,0.751369,0.171171,0.093596,1.000000,0.233333,0.150000
8,Stacked LSTM,RECESS_PERIOD,0.721839,0.196721,0.116505,0.631579,0.591667,0.700000
